<a href="https://colab.research.google.com/github/Amrutasai/azure-hello-world/blob/main/Telegram_Matrimony_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install beautifulsoup4 openpyxl
!pip install beautifulsoup4 pandas openpyxl

In [ ]:
import re
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import glob

# ---------- CONFIG ----------
MAX_AGE = 27
MIN_SALARY = 15
TARGET_CITIES = ["mumbai", "thane", "navi mumbai"]

CURRENT_YEAR = datetime.now().year


# ---------- HELPERS ----------

def extract_year(text):
    match = re.search(r'(19\d{2}|20\d{2})', text or "")
    return int(match.group(1)) if match else None


def calculate_age(year):
    return CURRENT_YEAR - year if year else None


def extract_dob(text):
    patterns = [
        r'DOB[:\s\-]*([^\n]+)',
        r'DoB[:\s\-]*([^\n]+)',
        r'Date of birth[:\s\-]*([^\n]+)',
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return None


def extract_salary(text):
    text = text.lower()
    match = re.search(r'(\d+(\.\d+)?)\s*(lpa|lac|lakh)', text)
    return float(match.group(1)) if match else None


def extract_name(text):
    match = re.search(r'Name[:\s\-]*([A-Za-z\s\.]+)', text, re.IGNORECASE)
    return match.group(1).strip() if match else None


def extract_contact(text):
    phones = re.findall(r'\b\d{10}\b', text)
    emails = re.findall(r'[\w\.-]+@[\w\.-]+', text)
    return ", ".join(phones), ", ".join(emails)


def extract_location(text):
    text_lower = text.lower()
    for city in TARGET_CITIES:
        if city in text_lower:
            return city.title()
    return None


def format_message(text_div):
    for br in text_div.find_all("br"):
        br.replace_with("\n")

    text = text_div.get_text()

    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]

    return "\n".join(lines)


# ---------- FINAL FILTER LOGIC ----------

def is_female_profile(name, text):
    text_lower = text.lower()

    # ❌ Name-based filter
    if name:
        name_lower = name.lower().strip()
        if name_lower.startswith(("ms ", "smt ", "ms. ", "smt. ")):
            return True

    # ❌ Emoji-based filter
    female_emojis = ["👩", "👰", "👧", "🙎‍♀️", "👩🏻", "👩🏻‍💼", "👩🏻‍⚖️"]
    if any(e in text for e in female_emojis):
        return True

    # ❌ "BRIDE DETAILS:" filter (case-insensitive, flexible)
    if re.search(r'bride\s*details[:\-\.]?', text_lower):
        return True

    if re.search(
    r'the\s+information\s+given\s+by\s+me\s+about\s+my\s+daughter\s+is\s+true',
    text_lower,
    re.IGNORECASE
    ):
      return True

    if re.search(
    r'\bname\s*of\s*(the\s*)?bride\b\s*[:\-]?',
    text_lower,
    re.IGNORECASE
):
      return True

    return False


# ---------- PROCESS SINGLE FILE ----------

def process_html(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    results = []

    for msg in soup.find_all("div", class_="message"):
        text_div = msg.find("div", class_="text")

        if not text_div:
            continue

        text = format_message(text_div)

        # Only groom posts
        if "groom" not in text.lower():
            continue

        name = extract_name(text)

        # ❌ Remove female profiles
        if is_female_profile(name, text):
            continue

        dob_text = extract_dob(text)
        year = extract_year(dob_text)
        age = calculate_age(year)

        salary = extract_salary(text)
        location = extract_location(text)
        phones, emails = extract_contact(text)

        # ---------- STRICT FILTER ----------
        if age is None or age > MAX_AGE:
            continue

        if salary is not None and salary < MIN_SALARY:
            continue

        if location is None:
            continue

        results.append({
            "Name": name,
            "DOB": dob_text,
            "Age": age,
            "Salary (LPA)": salary,
            "Location": location,
            "Phones": phones,
            "Emails": emails,
            "Message": text,
            "Source File": file_path
        })

    return results


# ---------- PROCESS ALL FILES ----------

files_list = glob.glob("message*.html")

print("Files detected:", files_list)

all_results = []

for file in files_list:
    print(f"Processing: {file}")
    file_results = process_html(file)
    all_results.extend(file_results)


# ---------- DEDUPLICATION ----------

seen = set()
unique_results = []

for row in all_results:
    key = (row["Name"], row["Phones"]) if row["Phones"] else (row["Name"],)

    if key not in seen:
        seen.add(key)
        unique_results.append(row)

data = unique_results


# ---------- OUTPUT ----------

print(f"\n✅ Total Unique Matches: {len(data)}\n")

for i, row in enumerate(data, 1):
    print(f"--- Match {i} ---")
    print(f"Name: {row['Name']}")
    print(f"Age: {row['Age']}")
    print(f"Salary: {row['Salary (LPA)']}")
    print(f"Location: {row['Location']}")
    print(f"Source: {row['Source File']}")
    print("\n--- Message ---")
    print(row["Message"])
    print("\n" + "="*80 + "\n")


# ---------- EXPORT ----------
df = pd.DataFrame(data)
df.to_excel("final_matches.xlsx", index=False)

print("✅ Excel created: final_matches.xlsx")

In [ ]:
from google.colab import files
files.download("final_matches.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>